# SETU — SeqKD vs DPO-distill on Google Colab (resumable)

The paper's head-to-head: same 52M student trained two ways, evaluated on the
**same real-reference dev set**.
- **S1 SeqKD** — SFT on the teacher's translations (Kim & Rush 2016 baseline).
- **S2 DPO-distill (ours)** — SFT on human references + DPO on ChrF-ranked preferences.

**Setup:** Runtime → Change runtime type → **GPU** (T4 is fine; A100/L4 on Pro is faster).
Then **Runtime → Run all**. A Google-Drive auth popup appears (cell 2).

**Resume after a disconnect — just Run All again.** Every expensive step (data,
preferences, the distilled corpus, each report) is saved to your Drive at
`MyDrive/setu_seqkd/` and marked **DONE** only when it fully succeeds. Cell 2 prints
a status list; any step already **DONE** is skipped in seconds, so you never
regenerate data or re-distill. So a 100k data + prefs + distill you already paid
for is *never* redone — the notebook picks up at the first unfinished step.
(The teacher distill uses greedy `--beams 1` so the whole run fits a session.)

In [ ]:
import torch
print('cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > GPU')
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/setu_seqkd'   # everything persists here across sessions
os.makedirs(WORK, exist_ok=True)
LIMIT = 100000                               # config — keep identical across S0-S3

# resume status — what's already saved on Drive (a step runs only if unchecked)
print('persisting artifacts to', WORK, '\n')
_steps = [('data',    '.done_data'),
          ('prefs',   '.done_prefs'),
          ('distill', '.done_distill'),
          ('S1 report', 'report_S1_seqkd.json'),
          ('S2 report', 'report_S2_dpo.json')]
for name, marker in _steps:
    print(f"  [{'DONE' if os.path.exists(f'{WORK}/{marker}') else '  - '}] {name}")

In [ ]:
# clone + install (code is ephemeral; only data/reports need to persist)
%cd /content
!rm -rf /content/SETU_v2
!git clone https://github.com/GeekyRiolu/SETU_v2.git
%cd /content/SETU_v2/SETU
!pip -q install -e ".[data,teacher,prefs,quantize]"
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml
# point data/ at Drive so the distilled corpus + data survive a disconnect
import os
os.makedirs(f'{WORK}/data', exist_ok=True)
!rm -rf data && ln -s {WORK}/data data
print('data ->', os.path.realpath('data'))

In [ ]:
# data + preferences. The `&& touch .done_*` marker is written ONLY on success,
# so a step interrupted mid-write is NOT falsely skipped — it re-runs cleanly.
import os
if not os.path.exists(f'{WORK}/.done_data'):
    !setu-data --limit {LIMIT + 2000} && touch {WORK}/.done_data
else:
    print('data already saved - skipping')
if not os.path.exists(f'{WORK}/.done_prefs'):
    !setu-prefs --max-entries 8000 && touch {WORK}/.done_prefs
else:
    print('prefs already saved - skipping')

In [ ]:
# teacher-distilled corpus (greedy = fast). Marker written only on success.
import os
if not os.path.exists(f'{WORK}/.done_distill'):
    !setu-distill --limit {LIMIT} --batch-size 32 --beams 1 && touch {WORK}/.done_distill
else:
    print('distilled corpus already saved - skipping')
_dist = 'data/distilled/hin_Deva-eng_Latn/train.jsonl'
assert os.path.exists(_dist) and os.path.getsize(_dist) > 0, 'distill produced no corpus - see output above'
print('distilled rows:', sum(1 for _ in open(_dist)))

In [ ]:
# S1 SeqKD: SFT on teacher targets, no DPO. Eval on real references.
if not os.path.exists(f'{WORK}/report_S1_seqkd.json'):
    !python scripts/train_full.py --train-corpus distilled --skip-dpo --limit {LIMIT} --dev-size 500 \
        && cp checkpoints/hin_Deva-eng_Latn/train_report.json {WORK}/report_S1_seqkd.json \
        && echo '=== S1 report saved to Drive ===' || echo '=== S1 FAILED - see output above ==='
else:
    print('S1 already done:', f'{WORK}/report_S1_seqkd.json')

In [ ]:
# S0 SFT (human refs) + S2 SFT+DPO. Same size, same dev set.
if not os.path.exists(f'{WORK}/report_S2_dpo.json'):
    !python scripts/train_full.py --train-corpus processed --limit {LIMIT} --dev-size 500 \
        && cp checkpoints/hin_Deva-eng_Latn/train_report.json {WORK}/report_S2_dpo.json \
        && echo '=== S2 report saved to Drive ===' || echo '=== S2 FAILED - see output above ==='
else:
    print('S2 already done:', f'{WORK}/report_S2_dpo.json')

In [ ]:
# comparison table (defensive)
import json, os
def load(p): return json.load(open(p)) if os.path.exists(p) else None
s1 = load(f'{WORK}/report_S1_seqkd.json'); s2 = load(f'{WORK}/report_S2_dpo.json')
if s1 is None: print('MISSING S1 report - the S1 cell did not finish')
if s2 is None: print('MISSING S2 report - the S2 cell did not finish')
def row(name, ev):
    r = ev.get('bleu_ratio')
    print(f"{name:26s} BLEU {ev['bleu']:6.2f}  chrF {ev['chrf']:6.2f}  ratio {r if r is None else round(r,3)}")
tb = (s2 or s1 or {}).get('sft_eval', {}).get('teacher_bleu')
print(f"\nteacher dev BLEU = {tb}\n")
if s1: row('S1 SeqKD (SFT-teacher)', s1['sft_eval'])
if s2: row('S0 SFT (human refs)', s2['sft_eval'])
if s2 and 'dpo_eval' in s2: row('S2 SFT-ref + DPO (ours)', s2['dpo_eval'])
print('\nPaste these into docs/PAPER_PLAN.md Table 1.')